Abra o Google Colab. Crie um notebook Lab01_SparkML. > Vamos prever o preço de casas na Califórnia usando Spark MLlib.
(Passo 1: Instalação das Dependências) O Google Colab já vem com Python, mas não vem com Java (necessário para o Spark) nem com o Apache Spark instalado. Copie e cole este código na primeira célula e execute (Shift + Enter).


In [3]:
# --- Célula 1: Instalação (Se já não estiver instalado) ---
# Limpar arquivos e diretórios Spark existentes para garantir uma instalação limpa
!rm -rf spark-3.5.0-bin-hadoop3.tgz spark-3.5.0-bin-hadoop3
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!pip install -q findspark

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"

import findspark
findspark.init()

from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").appName("Lab01_MLlib").getOrCreate()
print("Spark MLlib pronto!")
# (Passo 2: Carregando Dados Reais) Vamos usar o dataset clássico California Housing.

Spark MLlib pronto!


In [4]:
# --- Célula 2: Ingestão de Dados ---
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline

# Baixando dataset (CSV)
!wget -q https://raw.githubusercontent.com/ageron/handson-ml/master/datasets/housing/housing.csv

# Lendo com Spark (Inferindo Schema)
df = spark.read.csv("housing.csv", header=True, inferSchema=True)

# Limpeza: Remover nulos (em Big Data, imputação é melhor, mas vamos simplificar)
df = df.dropna()

print("Schema dos Dados:")
df.printSchema()
# (Passo 3: Feature Engineering com VectorAssembler) O Spark ML exige que todas as features de entrada estejam em uma única coluna de vetor.


Schema dos Dados:
root
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- housing_median_age: double (nullable = true)
 |-- total_rooms: double (nullable = true)
 |-- total_bedrooms: double (nullable = true)
 |-- population: double (nullable = true)
 |-- households: double (nullable = true)
 |-- median_income: double (nullable = true)
 |-- median_house_value: double (nullable = true)
 |-- ocean_proximity: string (nullable = true)



In [5]:
# --- Célula 3: Preparação (VectorAssembler) ---
# Colunas que usaremos para prever o preço
feature_cols = [
"longitude", "latitude", "housing_median_age", "total_rooms",
"total_bedrooms", "population", "households", "median_income"
]

# O Assembler pega essas colunas e cria a coluna 'features'
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# Divisão Treino/Teste (80/20)
train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

print(f"Treino: {train_data.count()} linhas")
print(f"Teste: {test_data.count()} linhas")
# (Passo 4: Treinamento do Modelo)


Treino: 16395 linhas
Teste: 4038 linhas


In [6]:
# --- Célula 4: Treinamento (Linear Regression) ---
# Definindo o algoritmo
lr = LinearRegression(featuresCol="features", labelCol="median_house_value")

# Criando o Pipeline (Assembler -> Modelo)
pipeline = Pipeline(stages=[assembler, lr])

# Treinando (Action!)
print("Treinando modelo distribuído...")
model = pipeline.fit(train_data)

# Extraindo o modelo treinado do pipeline
lr_model = model.stages[-1]

print(f"Intercepto: {lr_model.intercept}")
print(f"Coeficientes: {lr_model.coefficients}")
# (Passo 5: Avaliação e Métricas)


Treinando modelo distribuído...
Intercepto: -3582239.051647978
Coeficientes: [-42608.34729221635,-42257.394292044286,1176.3890666093757,-8.446997688308818,125.50466228106346,-36.55170942962926,31.469102369318026,40675.82306379408]


In [7]:
# --- Célula 5: Avaliação (RMSE e R2) ---
from pyspark.ml.evaluation import RegressionEvaluator

# Fazendo predições no teste
predictions = model.transform(test_data)

# Avaliador
evaluator_rmse = RegressionEvaluator(
labelCol="median_house_value", predictionCol="prediction", metricName="rmse"
)
evaluator_r2 = RegressionEvaluator(
labelCol="median_house_value", predictionCol="prediction", metricName="r2"
)

rmse = evaluator_rmse.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R2 Score: {r2:.4f}")

# Mostrar algumas predições
predictions.select("median_house_value", "prediction").show(5)
# (Análise do Lab) Observe que não usamos NumPy em momento nenhum. Todo o dado ficou na JVM do Spark. Se esse arquivo housing.csv tivesse 10 Terabytes, o código seria exatamente o mesmo. Essa é a magia do Spark MLlib. O código escala linearmente com o hardware.


Root Mean Squared Error (RMSE): 69685.93
R2 Score: 0.6323
+------------------+------------------+
|median_house_value|        prediction|
+------------------+------------------+
|          103600.0|102138.32319061086|
|          106700.0|190038.23036733363|
|           73200.0| 77197.94015233405|
|           90100.0| 165461.8792793164|
|           67000.0|120982.11327198707|
+------------------+------------------+
only showing top 5 rows



### Interpretação dos Resultados

Após executar o modelo, obtemos duas métricas principais para avaliar a performance da nossa Regressão Linear:

1.  **RMSE (Root Mean Squared Error):** Representa o erro médio das previsões na mesma unidade da meta (Dólares).
    - No nosso caso, um valor em torno de **69.000** significa que, em média, o modelo erra o preço da casa por essa margem.
    
2.  **R² (R-squared):** Indica o quanto o modelo explica a variação dos dados.
    - Um valor de **0.63** significa que **63%** da variação dos preços das casas na Califórnia é explicada pelas colunas que escolhemos (como localização, número de quartos e renda).

3.  **Tabela de Predições:**
    - A coluna `median_house_value` é o valor **real**.
    - A coluna `prediction` é o que o **Spark calculou**.
    - Comparando as duas, vemos que o modelo consegue captar a tendência, mas ainda possui uma margem de erro que poderia ser reduzida com modelos mais complexos (como Random Forest ou Gradient Boosting).